# Tableau Bulk Add Users
Interactive front end for the reusable helpers. Credentials/server details are loaded from persistent Windows environment variables; run inputs are prompted interactively.

In [ ]:
import importlib

import helpers.environment as _environment
import helpers.logging_setup as _logging_setup
import helpers.models as _models
import helpers.user_input as _user_input
import helpers.tableau_service as _tableau_service
import helpers.orchestrator as _orchestrator

# Jupyter keeps imported modules in memory. Reload helpers from disk so a git pull
# cannot leave this notebook bound to an older helper API from the same kernel.
for _module in (
    _models,
    _environment,
    _logging_setup,
    _user_input,
    _tableau_service,
    _orchestrator,
):
    importlib.reload(_module)

ensure_environment = _environment.ensure_environment
configure_logging = _logging_setup.configure_logging
resolve_and_read_users = _user_input.resolve_and_read_users
resolve_group_name = _user_input.resolve_group_name
TableauGroupService = _tableau_service.TableauGroupService
BulkAddOrchestrator = _orchestrator.BulkAddOrchestrator


## 1. Configure this run
The group name is prompted. The users-file prompt accepts a path; press Enter to use project-root `users.csv` or `users.txt` when present.

In [ ]:
GROUP_NAME = resolve_group_name(interactive=True)
USERS_FILE, USERS = resolve_and_read_users(interactive=True)
DRY_RUN = True
print(f"Users file: {USERS_FILE}")
print(f"Requested users: {', '.join(USERS)}")


## 2. Load or bootstrap Tableau authentication
If required values are missing, hidden prompts are used for secrets and the values are persisted in the Windows user environment.

In [ ]:
env = ensure_environment(interactive=True)
print(f"Environment ready: auth={env.auth_mode}, verify_ssl={env.verify_ssl}")


## 3. Run the orchestrator
This prints every intermediate stage and returns a structured report.

In [ ]:
log_file = configure_logging(verbose=True)
print(f"Log file: {log_file}")
service = TableauGroupService(env)
report = BulkAddOrchestrator(service).run(group_name=GROUP_NAME, users=USERS, dry_run=DRY_RUN)
report


## 4. Inspect the per-user result


In [ ]:
[(item.username, item.status.value, item.detail) for item in report.results]
